# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# .metadata is a Metadata object, printing as JSON for readability
metadata = dataset.metadata.to_json()
print('Dataset Name:', metadata['name'])
print('Description:', metadata['description'])

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# The Croissant metadata defines one or more record sets.
# Let's print record set @ids and their corresponding fields and @id for exploration.
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet: @id={rs['@id']} | Name: {rs.get('name', '')}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            if isinstance(field, dict):
                print(f"  Field: @id={field['@id']} | Name: {field.get('name', '')}")
            else:
                print(f"  Field: @id={field}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# --- Find available record_set @id(s) ---
# We'll extract all records from all record sets for demonstration.

# Croissant uses @id's to reference entities
record_sets = dataset.metadata.record_sets
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = dict()

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f'Records loaded for RecordSet @id: {record_set_id}. Shape: {df.shape}')

# Example: show columns for the first record set (if available)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns in main RecordSet (@id={main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print('No record sets available for extraction.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For illustration, select a numeric field from the main RecordSet. Adjust the field '@id' based on overview output.

if record_set_ids:
    main_rs_id = record_set_ids[0]
    df = dataframes[main_rs_id]

    # Try to pick a likely numeric field:
    numeric_candidates = [c for c in df.columns if df[c].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[c])]
    print(f"Numeric candidates: {numeric_candidates}")

    if len(numeric_candidates) > 0:
        numeric_field = numeric_candidates[0]

        threshold = df[numeric_field].mean() if not pd.isna(df[numeric_field].mean()) else 0
        print(f'Filtering {numeric_field} > {threshold:.2f} (mean)')
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical field
        possible_groups = [c for c in df.columns if c != numeric_field and df[c].nunique() < min(10, len(df)//5)]
        if possible_groups:
            group_field = possible_groups[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print('No suitable group field found.')
    else:
        print('No numeric fields found in main record set.')
else:
    print('No record sets available for analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Basic visualization: histogram for the chosen numeric field, if available
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and len(dataframes) > 0:
    main_rs_id = record_set_ids[0]
    df = dataframes[main_rs_id]
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()
    else:
        print('No numeric field available to plot.')
else:
    print('No data available for plotting.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.